# Large-Scale Portfolio Briefs Generation

## Step 1: Load Company Identifiers from CSV File

This first step reads the CSV file containing company information and extracts all unique company identifiers (RP_ENTITY_ID). These identifiers are used to specify which companies to generate briefs for.

**What it does:**
- Opens the CSV file with company data
- Finds the column containing company IDs
- Removes any empty or duplicate entries
- Creates a clean list of company identifiers for processing

**Output:** A list of unique company IDs that will be used in subsequent steps.


**Note:** All required dependencies (pandas, requests, xlsxwriter, ipython, jupyterlab) should be installed from `requirements.txt` before running this notebook. See the README for installation instructions.


In [1]:
# Read CSV and produce comma-separated RP_ENTITY_ID string
import pandas as pd
import json

df = pd.read_csv("static/data/US_100.csv", dtype=str)

col = next((c for c in df.columns if c.strip().upper() == "RP_ENTITY_ID"), None)
if col is None:
    raise ValueError("RP_ENTITY_ID column not found in static/data/US_100.csv")

ids = (
    df[col]
    .astype(str)
    .str.strip()
    .replace({"": None})
    .dropna()
    .drop_duplicates()
    .tolist()
)

print(len(ids))
#print(json.dumps(ids))
# rp_entity_ids_csv now contains the comma-separated IDs

5


## Step 2: Define Search Phrases (Topics)

This step defines short phrases used like web-style semantic search queries for each company. They steer which information is retrieved and summarized.

**What it does:**
- Sets up keyword-rich phrases customized with the company name (`{entity}`)
- Covers earnings and metrics, guidance, strategy, contracts, and products

**Note:** Adjust wording to match how your briefing service interprets search-style topics.


In [2]:
# Build request payload (adjust dates/topics as needed)
TOPICS = [

    'What material acquisition, or merger, takeover transactions involve {entity}, whether as buyer, seller, or target, and what are the key announced terms (valuation, consideration, ownership changes)?',

    'Have any investors or shareholders recently sold a controlling or significant minority stake in {entity} to a new financial or strategic buyer, and what are the implications for ownership and governance?',

    'Has {entity} been the subject of any sale, auction, or strategic alternatives process, including indications of interest, LOIs, or signed sale agreements?',

    'What notable bolt-on, tuck-in, or add-on acquisitions has {entity} announced or completed recently, and how do these transactions affect its scale and strategy?',

    'What divestiture, carve-out, spin-off, or asset sale transactions has {entity} announced or executed, and what businesses or assets are included in the perimeter?',

    'Is {entity} disposing of non-core business units, product lines, or regional operations, and what are the rationale, buyers, and expected closing timelines?',

    'Has {entity} agreed to transfer a portfolio company, business segment, or joint venture interest to another sponsor, fund, or strategic acquirer?',

    'Have there been any secondary sales of {entity} shares or fund interests, including GP- or LP‑led transactions, recapitalizations, or continuation vehicles affecting {entity}’s ownership?',

    'What new liquidity events have occurred for early investors, founders, or employees of {entity}, such as partial exits, structured secondaries, or tender offers?',

    'Has {entity} raised capital in a round that included significant secondary share sales, and how did this alter the cap table and control dynamics?',

    'Has {entity} filed for, announced, or completed an IPO, direct listing, SPAC merger, or other public-listing transaction, and what is the implied valuation and free float?',

    'Have there been any take‑private, de‑listing, or public‑to‑private transactions involving {entity}, and what are the key financing and ownership details?',

    'What recapitalization, dividend recap, or leveraged refinancing transactions has {entity} executed that provided liquidity to existing shareholders?',

    'Has {entity} issued new equity or hybrid securities in transactions that allowed existing investors to monetize or partially exit positions?',

    'What changes in the ownership of funds or vehicles that hold {entity} (e.g., GP stakes, fund secondaries, continuation funds) could impact the ultimate sponsor or decision‑makers for {entity}?',

    'Have any new sponsors, co‑investors, or strategic partners acquired stakes in the fund or holding vehicles that own {entity}, potentially signaling shifts in oversight or exit plans?',

    'What significant upcoming corporate events, market rumors, or regulatory developments could lead to M&A, divestiture, or liquidity events for {entity} in the near term?',

    'What material merger or acquisition transactions involve {entity}, whether as buyer, seller, or target, and what are the key announced terms (valuation, consideration, and change in control)?',

    'Has {entity} initiated or been the subject of a formal sale process, strategic alternatives review, or auction that could lead to a merger or acquisition?',

    'What signs of financial stress or uncertainty are emerging at {entity}, including going‑concern warnings, covenant breaches, payment delays, or auditor/emphasis‑of‑matter commentary?',

    'What actions is {entity} taking to address liquidity pressure or refinancing risk, such as cost‑cutting, asset sales, debt renegotiations, or standstill agreements?',

    'Has {entity} filed for, or publicly considered, bankruptcy, insolvency, administration, receivership, or formal restructuring proceedings, and what are the implications for creditors and shareholders?',

    'What out‑of‑court restructuring, distressed exchange, or turnaround plans has {entity} announced, and how do these affect its capital structure and ownership?',

    'What significant equity or debt fundraising transactions has {entity} completed or announced recently, including size, type of instrument, key investors, and implied valuation?',

    'Has {entity} executed any major recapitalization, dividend recap, or refinancing that provides liquidity to existing shareholders or materially alters leverage?',

    'Has {entity} filed for, announced, or completed an IPO, direct listing, SPAC merger, or other public listing, and what are the key details around valuation, free float, and use of proceeds?',

    'Have there been any take‑private, de‑listing, or public‑to‑private transactions involving {entity}, and what changes in ownership or control result from these deals?',

    'What divestiture, carve‑out, spin‑off, or business/asset sale transactions has {entity} announced, including which units are sold, who the buyers are, and the strategic rationale?',

    'Is {entity} disposing of non‑core or underperforming operations to raise liquidity or sharpen strategic focus, and what is the expected impact on its financial profile?',

    'What significant joint venture, strategic alliance, or co‑investment agreements involve {entity}, and how do these affect its capital commitments, risk‑sharing, or market access?',

    'Has {entity} contributed assets, IP, or business lines into a joint vehicle or partnership that creates partial monetization or shared ownership of those assets?',

    "What key results, guidance, and themes did {entity}'s latest earnings report and call highlight?",
 
    'What transfer restrictions or repurchase rights apply to securities issued in connection with a merger or acquisition involving {entity}?',
 
    'What changes to the certificate of incorporation, bylaws, or governance documents of {entity} are triggered by a pending or completed merger?',

    'What dual-class or multi-class share structure has {entity} implemented in connection with its IPO, and what are the voting rights and conversion mechanics for each class?',
 
    'What equity exchange right agreements or put right arrangements have {entity} and its co-founders entered into in connection with a public listing, and how do these affect founder control?',
 
    'What lock-up, market stand-off, or transfer restriction agreements have key shareholders and founders of {entity} entered into in connection with its IPO or public listing?',
 
    'What analyst coverage initiations, price targets, and ratings has {entity} received following its IPO or public listing?',
    ]



## Step 3: Configure Batch Processing and API Settings

This step sets up the configuration for generating briefs, including how many companies to process at once and the date range for the reports.

**What it does:**
- **Batch Size:** Determines how many companies are processed together (20 companies per batch for this example)
- **Company Selection:** Selects the first 100 companies from the list (can be changed to process all companies)
- **Date Range:** Sets the time period for the briefing (start and end dates)
- **API Configuration:** Sets up authentication and the API endpoint URL, if required
- **Processing Options:** Configures how the system prioritizes information (freshness, source ranking, novelty detection)

**Key Settings:**
- `BATCH_SIZE`: Number of companies processed per request (recommended: 50 for production)
- `report_start_date` and `report_end_date`: The time window for gathering information
- `novelty`: Whether to filter for only new or unique information
- `source_rank_boost` and `freshness_boost`: Control how sources are prioritized


In [3]:
import os
import json
import requests
import pandas as pd

# Batch size, go with 50 for prd use case
BATCH_SIZE = 50

# Take first 100
companies = ids

#for prd use case use 
#companies = ids

print("Using", len(companies), "companies")
#print(json.dumps(companies))

# service uses APIKeyQuery named `token` per your OpenAPI; set env var API_TOKEN (or TOKEN/API_KEY)
token = os.environ.get("API_TOKEN") or os.environ.get("TOKEN") or os.environ.get("API_KEY")
params = {"token": token} if token else {}

# v4+ / current API: "entities". Older images (pre-v4) required "companies" only.
ENTITY_BODY_FIELD = "entities"

payload = {
    ENTITY_BODY_FIELD: companies,
    "report_start_date": "2026-02-01",
    "report_end_date": "2026-02-28",
    "novelty": True,
    "sources": None,
    "topics": TOPICS,
    "source_rank_boost": 10,
    "freshness_boost": 8,
    "disable_introduction": True,

}

# Snapshot for exploratory grid / timeline if another cell reuses the name `payload`
BRIEF_CREATE_PAYLOAD = dict(payload)

# API call settings
# Update this to the actual create endpoint
API_URL = "http://localhost:8001/briefs/create"



Using 5 companies


## Step 4: Set Output Folder and File Names

All generated artifacts go under the **`output/`** directory (created if missing):
- **Request summary JSON:** batch request metadata (status, timestamps, entity counts)
- **Combined report JSON:** full briefing data and `source_metadata`
- **Excel export** (Step 10): uses the same folder via `OUT_XLSX`

Downstream cells read `BRIEF_REPORT_FILE` from this path after a successful batch run.


In [4]:
from pathlib import Path

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BRIEF_SUMMARY_FILE = OUTPUT_DIR / "briefs_request2_summaries1000.json"
BRIEF_REPORT_FILE = OUTPUT_DIR / "combined_briefs2_report1000.json"
OUT_XLSX = OUTPUT_DIR / "entities_bullets_1000.xlsx"

## Step 5: Batch brief generation

Sends `payload` to `/briefs/create` in chunks of `BATCH_SIZE`, polls status, merges `entity_reports` and `source_metadata`, then writes the two JSON files from Step 4. If some companies have no bullets after the first pass, a second pass retries them with extra topic `{entity} is`, `sentiment_threshold=0`, and `rerank_threshold=0.01` to boost recall.


In [5]:
import time
import copy
import json
import traceback
import requests
from datetime import datetime

current_datetime = datetime.now()
print(f"Batch Starting date and time: {current_datetime}")


def _status_url_for(request_id: str) -> str:
    # Update this to the actual status endpoint
    return f"http://localhost:8001/briefs/status/{request_id}"


def _normalize_entity_id(x: object) -> str:
    return str(x).strip()


def _rp_id_from_entity_report(er: dict) -> str | None:
    ei = er.get("entity_info") or {}
    rid = er.get("rp_entity_id") or er.get("entity_id") or ei.get("id")
    if rid is None or not str(rid).strip():
        return None
    return str(rid).strip()


def _entity_report_has_brief_content(er: dict) -> bool:
    # At least one non-empty bullet_point in content[]
    content = er.get("content") or []
    for item in content:
        if isinstance(item, dict) and str(item.get("bullet_point") or "").strip():
            return True
    return False


def _entity_ids_without_brief(all_company_ids: list, entity_reports: list) -> list:
    has_brief: set[str] = set()
    for er in entity_reports:
        rid = _rp_id_from_entity_report(er)
        if rid and _entity_report_has_brief_content(er):
            has_brief.add(rid)
    return [c for c in all_company_ids if _normalize_entity_id(c) not in has_brief]


def _run_brief_batches(
    entities: list,
    batch_payload: dict,
    *,
    pass_label: str,
) -> tuple[list, dict, dict]:
    tag = " (retry)" if pass_label == "retry" else ""
    entity_reports_out: list = []
    source_metadata_out: dict = {}
    summaries_out: dict = {}

    for start in range(0, len(entities), BATCH_SIZE):
        batch = entities[start : start + BATCH_SIZE]
        payload_batch = copy.deepcopy(batch_payload)
        payload_batch[ENTITY_BODY_FIELD] = batch

        try:
            print(
                f"Submitting batch {start + 1}-{start + len(batch)} ({len(batch)} entities){tag}..."
            )
            resp = requests.post(API_URL, params=params, json=payload_batch, timeout=180)
            resp.raise_for_status()
            create_resp = resp.json()
        except Exception as e:
            print("Create request failed for batch starting at", start, ":", e)
            traceback.print_exc()
            summaries_out[f"{pass_label}_batch_{start}"] = {
                "pass": pass_label,
                "start_date": payload_batch.get("report_start_date"),
                "end_date": payload_batch.get("report_end_date"),
                "logs": getattr(e, "args", str(e)),
                "report_title": None,
                "watchlist_id": None,
                "status": "create_failed",
                "rp_entity_ids": list(batch),
            }
            continue

        request_id = create_resp.get("request_id")
        immediate_report = create_resp.get("report", {}) or {}
        summary_key = request_id or f"{pass_label}_batch_{start}"
        summaries_out[summary_key] = {
            "pass": pass_label,
            "start_date": payload_batch.get("report_start_date"),
            "end_date": payload_batch.get("report_end_date"),
            "logs": create_resp.get("logs") or immediate_report.get("logs"),
            "report_title": immediate_report.get("report_title") or create_resp.get("report_title"),
            "watchlist_id": immediate_report.get("watchlist_id") or create_resp.get("watchlist_id"),
            "status": "submitted",
            "rp_entity_ids": list(batch),
        }

        if not request_id and immediate_report:
            ers = immediate_report.get("entity_reports", []) or []
            sm = immediate_report.get("source_metadata", {}) or {}
            entity_reports_out.extend(ers)
            source_metadata_out.update(sm)
            summaries_out[summary_key]["status"] = "completed_sync"
            print(f"Batch {start}-{start + len(batch)} returned sync report with {len(ers)} entities.{tag}")
            continue

        status_url = _status_url_for(request_id)
        timeout_seconds = 600
        poll_interval = 10
        waited = 0
        final_status_resp = None
        while waited < timeout_seconds:
            try:
                status_resp = requests.get(status_url, params=params, timeout=60)
                status_resp.raise_for_status()
                sjson = status_resp.json()
                status = sjson.get("status") or sjson.get("state") or ""
                if status and status.lower() in ("completed", "done", "success"):
                    final_status_resp = sjson
                    summaries_out[request_id]["status"] = "completed"
                    break
                if status and status.lower() in ("failed", "error"):
                    final_status_resp = sjson
                    summaries_out[request_id]["status"] = "failed"
                    break
            except Exception as e:
                print("Status check error:", e)
            time.sleep(poll_interval)
            waited += poll_interval

        if not final_status_resp:
            print(f"Timeout waiting for request {request_id}; proceeding to next batch.")
            summaries_out[request_id]["status"] = "timeout"
            continue

        report = final_status_resp.get("report", {}) or final_status_resp
        entity_reports_chunk = report.get("entity_reports", []) or []
        source_meta_chunk = report.get("source_metadata", {}) or {}

        entity_reports_out.extend(entity_reports_chunk)
        for k, v in source_meta_chunk.items():
            if k not in source_metadata_out:
                source_metadata_out[k] = v

        api_status = final_status_resp.get("status", "")
        summaries_out[request_id].update(
            {
                "api_status": str(api_status),
                "logs": final_status_resp.get("logs") or summaries_out[request_id].get("logs"),
                "report_title": report.get("report_title") or summaries_out[request_id].get("report_title"),
                "watchlist_id": report.get("watchlist_id") or summaries_out[request_id].get("watchlist_id"),
                "entity_count": len(entity_reports_chunk),
                "completed_at": final_status_resp.get("completed_at") or final_status_resp.get("ts"),
            }
        )

        print(f"Batch {start + 1}-{start + len(batch)} completed: {len(entity_reports_chunk)} entities added.{tag}")

        if len(entity_reports_chunk) == 0:
            if isinstance(report, dict) and report:
                intro = report.get("introduction") or ""
                if intro:
                    print("Report introduction (snippet):", str(intro)[:800])
            st = str(api_status).lower()
            if st in ("failed", "error"):
                print("Job FAILED — see logs in summary JSON.")
            elif st in ("completed", "done", "success"):
                print(
                    "No entity_reports in response: try wider dates, novelty=False, or broader TOPICS."
                )

    return entity_reports_out, source_metadata_out, summaries_out


combined_entity_reports, combined_source_metadata, request_summaries = _run_brief_batches(
    companies, payload, pass_label="primary"
)

for er in combined_entity_reports:
    ei = er.get("entity_info") or {}
    rid = er.get("rp_entity_id") or er.get("entity_id") or ei.get("id")
    if rid is not None and str(rid).strip():
        er["rp_entity_id"] = str(rid).strip()

missing_no_brief = _entity_ids_without_brief(companies, combined_entity_reports)
if missing_no_brief:
    print(
        f"Retry pass: {len(missing_no_brief)} entities without brief; "
        "adding topic '{entity} is', sentiment_threshold=0, rerank_threshold=0.01."
    )
    payload_retry = copy.deepcopy(payload)
    retry_topics = list(payload_retry.get("topics") or TOPICS)
    extra_topic = "{entity} is"
    if extra_topic not in retry_topics:
        retry_topics.append(extra_topic)
    payload_retry["topics"] = retry_topics
    payload_retry["sentiment_threshold"] = 0
    payload_retry["rerank_threshold"] = 0.01

    retry_reports, retry_sm, retry_summaries = _run_brief_batches(
        missing_no_brief, payload_retry, pass_label="retry"
    )
    missing_set = {_normalize_entity_id(m) for m in missing_no_brief}
    combined_entity_reports = [
        er for er in combined_entity_reports if _rp_id_from_entity_report(er) not in missing_set
    ]
    combined_entity_reports.extend(retry_reports)
    for k, v in retry_sm.items():
        if k not in combined_source_metadata:
            combined_source_metadata[k] = v
    request_summaries.update(retry_summaries)

    for er in combined_entity_reports:
        ei = er.get("entity_info") or {}
        rid = er.get("rp_entity_id") or er.get("entity_id") or ei.get("id")
        if rid is not None and str(rid).strip():
            er["rp_entity_id"] = str(rid).strip()
else:
    print("Retry pass skipped: all companies have at least one bullet from the primary run.")

combined_report = {
    "entity_reports": combined_entity_reports,
    "source_metadata": combined_source_metadata,
}

with open(BRIEF_REPORT_FILE, "w", encoding="utf-8") as f:
    json.dump(combined_report, f, ensure_ascii=False, indent=2)

with open(BRIEF_SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(request_summaries, f, ensure_ascii=False, indent=2)

print(
    f"Accumulated {len(combined_entity_reports)} entity_reports and "
    f"{len(combined_source_metadata)} source_metadata entries across {len(request_summaries)} requests."
)

current_datetime = datetime.now()
print(f"Batch Completion date and time: {current_datetime}")


Batch Starting date and time: 2026-03-31 13:03:28.291446
Submitting batch 1-5 (5 entities)...
Batch 1-5 completed: 1 entities added.
Retry pass: 4 entities without brief; adding topic '{entity} is', sentiment_threshold=0, rerank_threshold=0.01.
Submitting batch 1-4 (4 entities) (retry)...
Batch 1-4 completed: 1 entities added. (retry)
Accumulated 2 entity_reports and 7 source_metadata entries across 2 requests.
Batch Completion date and time: 2026-03-31 13:04:58.503790


## At this stage we should have briefing of selected companies

## Step 7: Set Up Display Functions for Notebook Viewing (Reference Purpose Only)

This step defines helper functions that format and display the briefing reports in a readable way within the Jupyter notebook. These functions are used later to show the results.

**What it does:**
- **`render_source_reference()`:** Formats source information (news articles, reports) with links and metadata
- **`present_entity_report()`:** Creates a nicely formatted display of a company's briefing report with:
  - Company name, sector, industry, and country
  - Numbered bullet points summarizing key information
  - Source links for each bullet point
  - Summary statistics

**Note:** These functions are defined here but used in the next step to display results.


In [ ]:
# For Presentation on Notebook 
from IPython.display import display, Markdown, HTML
import pandas as pd
import json
from pathlib import Path
from pprint import pprint
from typing import Tuple, Dict, Any, Optional
import html


def render_source_reference(
    source_id: str,
    source_metadata: Optional[Dict[str, Any]],
    show_highlights: bool = True,
    snippet_length: int = 300
) -> Tuple[str, Dict[str, Any]]:
    """
    Return (markdown_str, metadata_dict) for a given source id using the provided source_metadata map.
    - markdown_str: ready to display in a Jupyter cell via display(Markdown(...))
    - metadata_dict: the raw meta dict (for programmatic use)
    Behavior follows the selected lines: uses source_name/headline/url from the meta if present.
    """
    def _truncate(s: Optional[str], n: int) -> str:
        if not s:
            return ""
        return s if len(s) <= n else s[:n].rsplit(" ", 1)[0] + "…"

    if not source_metadata:
        md = f"`{source_id}` — (no source_metadata provided)"
        return md, {}

    meta = source_metadata.get(source_id) or {}
    if not meta:
        md = f"`{source_id}` — (not found in source_metadata)"
        return md, {}

    # chosen display fields (mirrors your selected snippet)
    name_or_headline = meta.get("source_name") or meta.get("headline") or source_id
    url = meta.get("url")
    ts = meta.get("ts")
    source_key = meta.get("source_key")
    text = meta.get("text")
    highlights = meta.get("highlights", [])

    # escape to avoid accidental HTML injection when rendering
    safe_name = html.escape(name_or_headline)
    safe_ts = html.escape(str(ts)) if ts else ""
    safe_key = html.escape(str(source_key)) if source_key else ""
    safe_snippet = html.escape(_truncate(text, snippet_length))

    # build markdown
    link_part = f"[{safe_name}]({html.escape(url)})" if url else f"**{safe_name}**"
    meta_parts = []
    if safe_ts:
        meta_parts.append(f"`{safe_ts}`")
    if safe_key:
        meta_parts.append(f"`{safe_key}`")
    meta_line = " • ".join(meta_parts)
    md_lines = [f"{link_part}  \n{meta_line}" if meta_line else f"{link_part}"]

    if safe_snippet:
        md_lines.append(f"\n> {safe_snippet}\n")

    if show_highlights and highlights:
        # highlights expected as list of {pnum:int, snum:int} or plain strings
        hl_lines = []
        for h in highlights[:6]:  # limit shown highlights
            if isinstance(h, dict):
                hl_lines.append(f"- paragraph {h.get('pnum')}, sentence {h.get('snum')}")
            else:
                hl_lines.append(f"- {html.escape(str(h))}")
        md_lines.append("**Highlights:**\n" + "\n".join(hl_lines))

    markdown = "\n\n".join(md_lines)
    return markdown, meta, link_part

def present_entity_report(entity: dict, source_metadata: dict | None = None, top_n: int | None = None):
    """
    Nicely render a single entity report for a financial analyst in a Jupyter notebook.
    - entity: the JSON object (keys: entity_id / rp_entity_id, entity_info, content)
    - source_metadata: optional dict mapping source_id -> metadata (url, source_name, headline)
    - top_n: limit number of bullet points shown
    """
    ei = entity.get("entity_info", {})
    name = ei.get("name", "Unknown")
    rp_entity_id = entity.get("rp_entity_id") or entity.get("entity_id") or ei.get("id") or "N/A"
    ticker = ei.get("ticker", "")
    sector = ei.get("sector", "—")
    industry = ei.get("industry", "—")
    country = ei.get("country", "—")
    webpage = ei.get("webpage")

    header = (
        f"## {name}\n"
        f"**RP_ENTITY_ID:** `{rp_entity_id}`  •  **Sector:** {sector}  •  **Industry:** {industry}  •  **Country:** {country}\n"
    )
    if webpage:
        header += f"[Website]({webpage})\n"
    display(Markdown(header))

    bullets = entity.get("content", []) or []
    if not bullets:
        display(Markdown("_No bullet points found for this entity._"))
        return

    # Summary metrics
    num_bullets = len(bullets)
    # gather source counts
    all_srcs = []
    for b in bullets:
        all_srcs.extend(b.get("sources", []))
    src_counts = pd.Series(all_srcs).value_counts()
    top_sources = src_counts.index.tolist()[:5]
    summary_md = f"**Bullet points:** {num_bullets}  •  **Top sources (ids):** {', '.join(top_sources) if top_sources else 'None'}\n"
    display(Markdown(summary_md))

    # Show bullets (numbered) with source links if metadata provided
    limit = top_n if top_n is not None else num_bullets
    for i, b in enumerate(bullets[:limit], start=1):
        text = b.get("bullet_point", "").strip()
        srcs = b.get("sources", []) or []
        # resolve sources to friendly links/names if metadata available
        resolved = []
        markdowns = []
        for s in srcs:
            if source_metadata and s in source_metadata:
                meta = source_metadata[s]
                name_or_headline = meta.get("source_name") or meta.get("headline") or s
                url = meta.get("url")
                if url:
                    resolved.append(f"[{name_or_headline}]({url})")
                else:
                    resolved.append(f"{name_or_headline} ({s})")
            else:
                resolved.append(s)

            markdown, meta, linkpart = render_source_reference(s, source_metadata)
            markdowns.append(linkpart)

        src_line = ", ".join(resolved) if resolved else "None"
        updated_src_line = ", ".join(markdowns) if markdowns else "None"
        display(Markdown(f"{i}. {text}\n\n**Sources:** {updated_src_line}\n"))

        
    # Provide a small table for quick export / analysis
    df = pd.DataFrame([
        {"rp_entity_id": rp_entity_id, "bullet": b.get("bullet_point", ""), "sources": b.get("sources", [])}
        for b in bullets
    ])
    display(Markdown("**Raw table (for copy/export):**"))
    display(df.head(limit))



## Step 8: Load Saved Briefing Report

This step loads the briefing report that was saved during batch processing. The report contains all company briefings and their source information.

**What it does:**
- Opens the saved JSON file containing the combined briefing report
- Extracts the company reports and source metadata
- Makes the data available for display or export in subsequent steps

**Output:** 
- `entities`: List of all company briefing reports
- `source_metadata`: Dictionary mapping source IDs to source information (URLs, headlines, publication dates)


In [7]:
#read entities and source_metadata from save file 
import json
from pathlib import Path
from pprint import pprint

REPORT_PATH = Path(BRIEF_REPORT_FILE)

if not REPORT_PATH.exists():
    raise FileNotFoundError(
        f"{REPORT_PATH} not found. Run Step 4 (output paths) then the batching cell that writes {BRIEF_REPORT_FILE.name} first."
    )

with REPORT_PATH.open("r", encoding="utf-8") as f:
    combined = json.load(f)

entities = combined.get("entity_reports", []) or []
source_metadata = combined.get("source_metadata", {}) or {}

print(f"Loaded combined report: {len(entities)} entities, {len(source_metadata)} source_metadata entries.")
# optional quick inspect
if entities:
    print("First entity keys:", list(entities[0].keys()))
if source_metadata:
    print("Sample source id:", next(iter(source_metadata.keys())))

Loaded combined report: 2 entities, 7 source_metadata entries.
First entity keys: ['entity_id', 'entity_info', 'content', 'rp_entity_id']
Sample source id: AD7C5F3B607D25601691E0F8C671D66F-54


## Step 9: Display Sample Reports in Notebook (Reference Purpose Only)

This step displays a preview of the briefing reports directly in the notebook. It shows the first 5 companies with their top 5 bullet points each.

**What it does:**
- Takes the first 5 companies from the loaded report
- Formats each company's briefing with:
  - Company information (name, sector, industry)
  - Key bullet points summarizing important information
  - Clickable source links for each bullet point
- Displays everything in a clean, readable format

**Purpose:** Allows you to review the briefing reports immediately in the notebook before exporting to other formats.


In [8]:
# Now we can present the report in a notebook
reportable_entities = entities[:3] # picking first 5 entities for presentation
for ent in reportable_entities:
    present_entity_report(ent, source_metadata=source_metadata, top_n=5)

## Ethos Technologies Inc.
**RP_ENTITY_ID:** `R49HQR`  •  **Sector:** Financials  •  **Industry:** Life Insurance  •  **Country:** US
[Website](http://www.ethos.com)


**Bullet points:** 2  •  **Top sources (ids):** AD7C5F3B607D25601691E0F8C671D66F-54, AD7C5F3B607D25601691E0F8C671D66F-12, AD7C5F3B607D25601691E0F8C671D66F-21


1. **Ethos Technologies Inc.** launches a new Indexed Universal Life product in partnership with a carrier, utilizing a unique distribution model that combines both teams to accelerate agent onboarding and market penetration.

**Sources:** **Factset Transcripts**


2. The company outlines three growth strategies for 2026: expanding its consumer base and agent recruitment, enhancing agent productivity, and broadening its product portfolio to capture a larger market share.

**Sources:** **Factset Transcripts**, **Factset Transcripts**


**Raw table (for copy/export):**

...

1. **Kodiak Building Partners Inc.** enters into a merger agreement with **QXO, Inc.** on February 10, 2026, involving **Juno Merger Sub, Inc.** as a wholly owned subsidiary, with the merger expected to finalize soon, impacting ownership and governance structures significantly.

**Sources:** [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/1236275/000110465926012932/tm265913d1_8k.htm), [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/1236275/000110465926012932/tm265913d1_8k.htm), [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/1236275/000110465926012932/tm265913d1_8k.htm)


2. The merger will result in the governance documents of **Kodiak Building Partners Inc.** being amended to align with those of **Juno Merger Sub, Inc.**, indicating a shift in operational control and potential strategic direction post-merger.

**Sources:** [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/1236275/000110465926012932/tm265913d1_8k.htm)


3. Post-merger, the name of the surviving entity will remain **Kodiak Building Partners Inc.**, but its certificate of incorporation and bylaws will reflect those of **Juno Merger Sub, Inc.**, suggesting a significant change in governance and operational oversight.

**Sources:** [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/1236275/000110465926012932/tm265913d1_8k.htm)


**Raw table (for copy/export):**

,rp_entity_id,bullet,sources
0,8C8XX4,**Kodiak Building Partners Inc.** enters into ...,"[076F71B842452D71EA6B0575B498B360-9, 076F71B84..."
1,8C8XX4,The merger will result in the governance docum...,[076F71B842452D71EA6B0575B498B360-168]
2,8C8XX4,"Post-merger, the name of the surviving entity ...",[076F71B842452D71EA6B0575B498B360-168]


## Step 10: Export Briefing Report to Excel (Reference Purpose Only)

This final step converts the briefing reports into an Excel spreadsheet format that can be easily shared, analyzed, or imported into other tools.

**What it does:**
- Creates a structured table with one row per bullet point
- Includes company information (name, sector, industry, country, website) on the first row for each company
- Lists bullet points with their associated sources
- Adds clickable hyperlinks to the source URLs in the Excel file
- Saves the workbook under `output/` (same folder as Step 4; see `OUT_XLSX`)

**Output:** An Excel file that can be opened in Microsoft Excel, Google Sheets, or any spreadsheet application. Each row contains a bullet point, and the source column includes clickable links to the original articles or reports.


In [9]:
# Export report to Excel 

import pandas as pd
import json
from IPython.display import display

# OUT_XLSX is set in Step 4 (under output/)

source_map = source_metadata

rows = []
link_meta = []  # parallel list of lists of urls (or None) for each row

for e1 in entities:
    # print (e1)
    ei = e1.get("entity_info", {}) or {}
    rp_entity_id = e1.get("rp_entity_id") or e1.get("entity_id") or ei.get("id") or ""
    name = ei.get("name") or ei.get("id") or e1.get("entity_id", "")
    sectors = ei.get("sector", "")
    industry = ei.get("industry", "")
    country = ei.get("country", "")
    website = ei.get("webpage", "") or ei.get("web_site", "")

    bullets = e1.get("content", []) or []
    if not bullets:
        rows.append({
            "rp_entity_id": rp_entity_id,
            "entity name": name,
            "sectors": sectors,
            "industry": industry,
            "country": country,
            "website": website,
            "bulletpoint": "",
            "source": ""
        })
        link_meta.append([])  # no links
        continue

    for i, b in enumerate(bullets):
        bp = b.get("bullet_point", "").strip()
        srcs = b.get("sources", []) or []

        resolved_displays = []
        resolved_urls = []
        for s in srcs:
            meta = source_map.get(s) or {}
            headline = meta.get("headline") or meta.get("source_name")
            url = meta.get("url")
            display_text = headline if headline else s
            resolved_displays.append(display_text)
            resolved_urls.append(url)  # may be None

        source_field = "; ".join(resolved_displays)
        first_url = next((u for u in resolved_urls if u), None)  # first available URL (or None)

        if i == 0:
            rows.append({
                "rp_entity_id": rp_entity_id,
                "entity name": name,
                "sectors": sectors,
                "industry": industry,
                "country": country,
                "website": website,
                "bulletpoint": bp,
                "source": source_field
            })
        else:
            rows.append({
                "rp_entity_id": "",
                "entity name": "",
                "sectors": "",
                "industry": "",
                "country": "",
                "website": "",
                "bulletpoint": bp,
                "source": source_field
            })

        link_meta.append([first_url])  # store first url (or [None])

df_out = pd.DataFrame(rows, columns=[
    "rp_entity_id", "entity name", "sectors", "industry", "country", "website", "bulletpoint", "source"
])

# Write to Excel with first source as hyperlink (cell displays all source texts, link opens first URL)

with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
    df_out.to_excel(writer, index=False, sheet_name="Briefs")
    workbook = writer.book
    worksheet = writer.sheets["Briefs"]

    # find source column index
    src_col = df_out.columns.get_loc("source")

    # rows in sheet start at 1 (0 is header)
    for r_idx, lm in enumerate(link_meta, start=1):
        first_url = lm[0] if lm else None
        if first_url:
            display_text = df_out.iloc[r_idx - 1]["source"] or first_url
            # write_url will display the provided string but link to first_url
            worksheet.write_url(r_idx, src_col, first_url, string=display_text)

print(f"Written {len(df_out)} rows to {OUT_XLSX}")
display(df_out.head(20))


Written 5 rows to output/entities_bullets_1000.xlsx


,rp_entity_id,entity name,sectors,industry,country,website,bulletpoint,source
0,R49HQR,Ethos Technologies Inc.,Financials,Life Insurance,US,http://www.ethos.com,**Ethos Technologies Inc.** launches a new Ind...,"Ethos Technologies, Inc.: Q4 2025 Earnings Call"
1,,,,,,,The company outlines three growth strategies f...,"Ethos Technologies, Inc.: Q4 2025 Earnings Cal..."
2,8C8XX4,Kodiak Building Partners Inc.,Industrials,Industrial Suppliers,US,,**Kodiak Building Partners Inc.** enters into ...,"QXO, Inc. files FORM 8-K on Feb 11, 2026; QXO,..."
3,,,,,,,The merger will result in the governance docum...,"QXO, Inc. files FORM 8-K on Feb 11, 2026"
4,,,,,,,"Post-merger, the name of the surviving entity ...","QXO, Inc. files FORM 8-K on Feb 11, 2026"
